# Fraud Detection Parquet: Data Review and Exploratory Analysis

This notebook reads the labeled training data only. It covers exploratory data analysis and label-free historical graph features. The external test data remain isolated until the model and threshold are finalized for inference.


In [ ]:
from pathlib import Path
import pandas as pd

# Jupyter may start the kernel in the notebook directory or the project root.
# Find the labeled training file and the submitted helper modules together.
PROJECT_ROOT = None
CODE_DIR = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    candidate_code = candidate / 'submission' / 'code'
    if not candidate_code.is_dir():
        candidate_code = candidate / 'code'
    if (candidate / 'train_fraud.parquet').is_file() and (candidate_code / 'plot').is_dir():
        PROJECT_ROOT = candidate
        CODE_DIR = candidate_code
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Place train_fraud.parquet beside the submission folder and open the notebook from that project.')
DATA_DIR = PROJECT_ROOT
TRAIN_PATH = DATA_DIR / 'train_fraud.parquet'


In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
print('Training dataset: train_fraud.parquet')
print(f'Shape: {train_df.shape}')
display(train_df.head())


In [ ]:
print('Training columns and data types')
display(train_df.dtypes.to_frame('dtype'))


In [ ]:
# Customer ID prefix meanings are documented in columns.txt. Only the training data are used here.
for role in ('initiating_customer', 'recipient_customer'):
    train_df[f'{role}_is_C'] = train_df[role].astype('string').str.startswith('C').astype('int8')
    train_df[f'{role}_is_M'] = train_df[role].astype('string').str.startswith('M').astype('int8')

new_columns = [
    'initiating_customer_is_C', 'initiating_customer_is_M',
    'recipient_customer_is_C', 'recipient_customer_is_M',
]
display(train_df[['initiating_customer', 'recipient_customer'] + new_columns].head())


In [ ]:
train_df

In [ ]:
# The external test data are read only during final inference; they are neither displayed nor used in this notebook.


In [ ]:
# Sort for temporal EDA. Customer IDs do not define the true order of transactions within the same elapsed hour.
train_df = train_df.sort_values(
    ['hours_elapsed', 'initiating_customer'], kind='stable'
)
display(train_df.head())


In [ ]:
# EDA helper functions are kept in code/plot to keep this Notebook readable.
import sys
from pathlib import Path

PLOT_CODE_DIR = CODE_DIR / 'plot'
if str(PLOT_CODE_DIR) not in sys.path:
    sys.path.append(str(PLOT_CODE_DIR))

from eda_plots import (
    get_dataset_quality_metrics,
    plot_label_distribution,
    plot_time_distribution,
    summarize_data_quality,
)

# Data-quality summary for the training data used in EDA/model development.
quality_metrics = get_dataset_quality_metrics(train_df)
print('Training-data quality metrics:')
display(pd.Series(quality_metrics, name='value'))
display(summarize_data_quality(train_df))

# Generate the two requested EDA figures. The external test set is not used here.
label_figure = plot_label_distribution(train_df, output_dir='image/data_quality_and_labels')
time_figure = plot_time_distribution(train_df, output_dir='image/time_distribution')
print(f'Saved label figure: {label_figure}')
print(f'Saved time figure: {time_figure}')

### Additional data-quality diagnostics

These two diagnostics use the training data only. The zero-value plot distinguishes observed zeros from missing values, while the cardinality plot shows which raw fields are high-cardinality and therefore need special handling. The four customer-type indicator columns are excluded from the cardinality plot because they were engineered earlier.

In [ ]:
from data_quality_eda import (
    plot_field_cardinality,
    plot_zero_value_rates,
    summarize_field_cardinality,
    summarize_zero_rates,
)
from IPython.display import IFrame

zero_value_summary = summarize_zero_rates(train_df)
field_cardinality_summary = summarize_field_cardinality(train_df, engineered_columns=new_columns)

print('Exact-zero summary by label:')
display(zero_value_summary)
print('Raw-field cardinality summary:')
display(field_cardinality_summary)

zero_value_figure = plot_zero_value_rates(
    train_df,
    output_dir='image/data_quality_and_labels',
)
field_cardinality_figure = plot_field_cardinality(
    train_df,
    output_dir='image/data_quality_and_labels',
    engineered_columns=new_columns,
)
print(f'Saved: {zero_value_figure}')
print(f'Saved: {field_cardinality_figure}')

# Preview the PDF figures directly in the notebook; no PNG files are created.
display(IFrame(src=str(zero_value_figure), width='100%', height=520))
display(IFrame(src=str(field_cardinality_figure), width='100%', height=650))

## Temporal distribution shift within the training data

This section compares three equal-width `hours_elapsed` windows in the training data. It examines transaction-type composition, transaction volume, fraud prevalence, and numeric distributions. The external test set is not used for these EDA calculations.
The following rule-stability plot separates rule hits, missed fraud and legitimate transaction volume in explicit hour windows.


In [ ]:
from distribution_shift_eda import (
    plot_numeric_distribution_shift,
    plot_temporal_composition_shift,
    summarize_temporal_shift,
)
from IPython.display import IFrame

distribution_shift = summarize_temporal_shift(train_df)
print('Temporal-window summary:')
display(distribution_shift['period_summary'])
print('Transaction-type composition by temporal window (%):')
display(distribution_shift['composition_pct'])
print(f"Early–Late transaction-type Jensen–Shannon distance: {distribution_shift['type_js_distance_early_late']:.4f}")
print('Numeric distribution-shift summary:')
display(distribution_shift['numeric_summary'])

temporal_composition_figure = plot_temporal_composition_shift(
    train_df,
    output_dir='image/distribution_shift',
)
numeric_distribution_figure = plot_numeric_distribution_shift(
    train_df,
    output_dir='image/distribution_shift',
)
print(f'Saved: {temporal_composition_figure}')
print(f'Saved: {numeric_distribution_figure}')
display(IFrame(src=str(temporal_composition_figure), width='100%', height=520))
display(IFrame(src=str(numeric_distribution_figure), width='100%', height=680))

In [ ]:
# Pre-transaction full-balance rule, shown by transaction type and hour window.
from full_balance_eda import plot_rule_over_time, summarize_rule_over_time

rule_time_summary = summarize_rule_over_time(train_df)
display(rule_time_summary)
rule_time_figure = plot_rule_over_time(train_df)
print(f'Saved: {rule_time_figure}')


## 1. Transaction type and fraud relationship

This block compares transaction volume, fraud count, and fraud rate by transaction type. It uses the training data only.

In [ ]:
from transaction_type_eda import (
    plot_transaction_type_counts,
    plot_transaction_type_fraud_rates,
    summarize_transaction_types,
)

transaction_type_summary = summarize_transaction_types(train_df)
display(transaction_type_summary)

transaction_type_count_figure = plot_transaction_type_counts(train_df)
transaction_type_rate_figure = plot_transaction_type_fraud_rates(train_df)
print(f'Saved: {transaction_type_count_figure}')
print(f'Saved: {transaction_type_rate_figure}')

## 2. Transaction amount and balance behavior

The amount distribution uses log1p(transaction_amount). Balance diagnostics are summarized separately by transaction type because the expected money flow differs across transaction types.
The following two plots isolate the positive-balance full-withdrawal rule and display amount-to-starting-balance bins with support counts and fraud rates.


In [ ]:
from amount_balance_eda import (
    plot_amount_balance_behavior,
    summarize_amounts,
    summarize_balance_behavior,
)

amount_summary = summarize_amounts(train_df)
balance_summary = summarize_balance_behavior(train_df)
display(amount_summary)
display(balance_summary['by_transaction_type'])

amount_balance_figure = plot_amount_balance_behavior(train_df)
print(f'Saved: {amount_balance_figure}')

In [ ]:
# Amount-to-starting-balance relationship; zero denominators stay separate.
# These are full-training-set EDA findings, not held-out model scores.
from full_balance_eda import (
    plot_balance_ratio_bins,
    plot_full_balance_rule,
    summarize_balance_ratio_bins,
    summarize_full_balance_rule,
)

full_balance_summary = summarize_full_balance_rule(train_df)
ratio_bin_summary = summarize_balance_ratio_bins(train_df)
display(full_balance_summary)
display(ratio_bin_summary)
full_balance_figure = plot_full_balance_rule(train_df)
ratio_bin_figure = plot_balance_ratio_bins(train_df)
print(f'Saved: {full_balance_figure}')
print(f'Saved: {ratio_bin_figure}')


## 3. Customer types and account history

The role-pair analysis uses the four binary customer-type features created earlier. Historical coverage is evaluated with strictly earlier hours; transactions in the same hour are not treated as ordered.

In [ ]:
from customer_history_eda import plot_customer_history, summarize_customer_history

customer_history = summarize_customer_history(train_df)
display(customer_history['role_summary'])
display(customer_history['history_metrics'])

print('Initiating-account activity summary:')
display(customer_history['initiator_activity'].describe(percentiles=[0.5, 0.75, 0.9, 0.99]).T)
print('Recipient-account activity summary:')
display(customer_history['recipient_activity'].describe(percentiles=[0.5, 0.75, 0.9, 0.99]).T)

customer_history_figure = plot_customer_history(train_df)
print(f'Saved: {customer_history_figure}')

## 4. Existing fraud-flagging system

is_flagged_fraud is evaluated as a pre-existing binary classifier against the actual is_fraud label.

In [ ]:
from system_flag_eda import plot_system_flag_performance, summarize_system_flag

system_flag_result = summarize_system_flag(train_df)
print('Confusion matrix:')
display(system_flag_result['confusion_matrix'])
print('Fraud-class metrics:')
display(system_flag_result['metrics'].to_frame())

system_flag_figure = plot_system_flag_performance(train_df)
print(f'Saved: {system_flag_figure}')

## GFP-inspired historical graph features

We represent each recorded transaction as a directed initiator-to-recipient edge. The twelve lightweight graph statistics below count only transactions in **strictly earlier hours**. They combine prior degree/fan, amount summaries, repeated-pair history, and a two-edge reciprocal history count. They are inspired by the vertex-statistics and temporal-cycle ideas in GFP; they are not Snap ML's full cycle/scatter-gather miner. No external test rows or fraud labels enter feature generation. The comparisons are descriptive training-set EDA, not validation performance.


In [ ]:
from pathlib import Path
import sys

GFP_CODE_DIR = CODE_DIR / 'GFP'
if str(GFP_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(GFP_CODE_DIR))

PLOT_CODE_DIR = CODE_DIR / 'plot'
if str(PLOT_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(PLOT_CODE_DIR))

from graph_features import build_gfp_features
from gfp_additional_features import build_additional_gfp_features
from gfp_topology_eda import (
    ALL_FEATURE_COLUMNS,
    plot_topology_enrichment,
    plot_topology_fraud_rates,
    plot_topology_label_distributions,
    summarize_topology_eda,
)

gfp_features = pd.concat(
    [build_gfp_features(train_df), build_additional_gfp_features(train_df)],
    axis=1,
)
assert tuple(gfp_features.columns) == ALL_FEATURE_COLUMNS
assert len(gfp_features) == len(train_df)
assert gfp_features.index.equals(train_df.index)
print('Training-only graph feature preview:')
display(gfp_features.head())

gfp_eda = summarize_topology_eda(train_df, gfp_features)
metrics_dir = Path('outputs/metrics')
metrics_dir.mkdir(parents=True, exist_ok=True)
feature_summary_path = metrics_dir / 'gfp_topology_feature_summary.csv'
binned_summary_path = metrics_dir / 'gfp_topology_binned_summary.csv'
gfp_eda['feature_summary'].to_csv(feature_summary_path, index=False)
gfp_eda['binned_summary'].to_csv(binned_summary_path, index=False)
display(gfp_eda['feature_summary'])
display(gfp_eda['binned_summary'].query("transaction_type in ['CASH_OUT', 'TRANSFER']"))

distribution_pdf = plot_topology_label_distributions(
    train_df, gfp_features, output_dir=str(Path('image') / 'topology_features')
)
rate_pdf = plot_topology_fraud_rates(
    train_df, gfp_features, output_dir=str(Path('image') / 'topology_features')
)
enrichment_pdf = plot_topology_enrichment(
    gfp_eda['feature_summary'], output_dir=str(Path('image') / 'topology_features')
)
# Fraud is concentrated late in the observed training period. This matched
# descriptive view checks whether topology separation remains in hours 635–718.
late_mask = train_df['hours_elapsed'].between(635, 718)
late_eda = summarize_topology_eda(
    train_df.loc[late_mask], gfp_features.loc[late_mask]
)
late_summary_path = metrics_dir / 'gfp_topology_feature_summary_hours_635_718.csv'
late_eda['feature_summary'].to_csv(late_summary_path, index=False)
late_distribution_pdf = plot_topology_label_distributions(
    train_df.loc[late_mask], gfp_features.loc[late_mask],
    output_dir=str(Path('image') / 'topology_features'),
    context='hours 635-718',
    filename='gfp_topology_label_distributions_hours_635_718',
)
print(f'Saved {feature_summary_path}, {binned_summary_path}, {distribution_pdf}, {rate_pdf}, {enrichment_pdf}, and {late_distribution_pdf}')